# Akan-BPE — Qwen3-1.7B × Akan TTS tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/run-qwen-1.7b.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/run-qwen-1.7b.ipynb)

QLoRA fine-tune of a base LLM with the Akan TTS tokenizer, scored against the unmodified base on **bits-per-byte (BPB, full byte coverage)**. Runs two arms — random and mean-of-subword embedding init — and compares them. Run top-to-bottom on a Kaggle/Colab **T4 GPU** (~45–50 min per arm).

## Setup

In [1]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Cloning into 'akan-bpe'...
remote: Enumerating objects: 658, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 658 (delta 88), reused 103 (delta 61), pack-reused 516 (from 1)
Receiving objects: 100% (658/658), 1.13 MiB | 17.59 MiB/s, done.
Resolving deltas: 100% (412/412), done.
/kaggle/working/akan-bpe
Working directory: /kaggle/working/akan-bpe


In [2]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.4 MB/s eta 0:00:00:00:0100:01


In [8]:
# Hugging Face authentication. Paste a token when prompted — it unblocks and
# speeds up the dataset downloads below (input is hidden).
from getpass import getpass

from huggingface_hub import login

login()

In [9]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a T4 GPU accelerator, then re-run.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [11]:
# Download Akan datasets. --tts-limit 50000 pins the 45,000/2,500/2,500 split.
!python scripts/download.py --output-dir data --tts-limit 50000

Resolving data files: 100%|████████████████| 270/270 [00:00<00:00, 25049.48it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 11.1MB/s]
Wrote 40000 rows to data/pristine_twi_train.jsonl
Wrote 5000 rows to data/pristine_twi_validation.jsonl
Wrote 5000 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [12]:
# Train the TTS tokenizer if not already present
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    print("TTS tokenizer already present, skipping.")

TTS tokenizer already present, skipping.


In [13]:
# Verify required inputs exist
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

All inputs ready.


## Run — random vs mean-of-subword embedding init

Defaults (eval cap, max-length, LoRA rank, learning rate, etc.) live in `scripts/model_integration.py`; pass `--help` to see or override them. Each arm writes its own adapter dir + result JSON derived from the model id.

In [14]:
# Arm A — random embedding init (default)
!python scripts/model_integration.py --model-id Qwen/Qwen3-1.7B-Base

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.58MB/s]
tokenizer_config.json: 9.68kB [00:00, 19.2MB/s]
vocab.json: 2.78MB [00:00, 51.1MB/s]
merges.txt: 1.67MB [00:00, 60.2MB/s]
tokenizer.json: 7.03MB [00:00, 143MB/s]
Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1695.41text/s]
model.safetensors: 100%|████████████████████| 3.44G/3.44G [00:14<00:00, 242MB/s]
Loading weights: 100%|█| 310/310 [00:01<00:00, 217.94it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 414kB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(ms

In [15]:
# Arm B — mean-of-subword embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-1.7B-Base --embedding-init-mode mean_subword

Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1833.00text/s]
Loading weights: 100%|█| 310/310 [00:01<00:00, 250.78it/s, Materializing param=m
Initializing mean-subword embeddings: 100%|█| 8000/8000 [00:01<00:00, 4246.28tok
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '6.826', 'grad_norm': '5.425', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.657', 'grad_norm': '3.251', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.332', 'grad_norm': '2.218', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.038', 'grad_norm': '2.544', 'learning_rat

## Results — compare the two arms

In [16]:
import json
from pathlib import Path

def load_eval(path):
    r = json.loads(Path(path).read_text(encoding="utf-8"))
    b = r["eval"]["bpb"]
    return {
        "init": r["embedding_init_mode"],
        "eval_loss": r["eval"]["eval_loss"],
        "perplexity": r["eval"]["perplexity"],
        "exp_bpb": b["experiment"]["bits_per_byte"],
        "base_bpb": (b["base"] or {}).get("bits_per_byte"),
        "improvement": b.get("improvement"),
    }

arms = {
    "random": load_eval("results/run-qwen-1.7b.json"),
    "mean_subword": load_eval("results/run-qwen-1.7b-meansub.json"),
}
header = f"{'metric':<24}{'random':>14}{'mean_subword':>16}"
print(header)
print("-" * len(header))

def show(label, key, fmt="{:.4f}"):
    cells = [fmt.format(arms[a][key]) if arms[a][key] is not None else "—"
             for a in ("random", "mean_subword")]
    print(f"{label:<24}{cells[0]:>14}{cells[1]:>16}")

show("eval_loss", "eval_loss")
show("perplexity", "perplexity", "{:.2f}")
show("experiment BPB", "exp_bpb")
show("base BPB", "base_bpb")
show("improvement (base−exp)", "improvement")

best = min(arms, key=lambda a: arms[a]["exp_bpb"])
print(f"\nLower fine-tuned BPB: {best!r} arm.")

metric                          random    mean_subword
------------------------------------------------------
eval_loss                       4.4109          3.7069
perplexity                       82.34           40.73
experiment BPB                  1.4741          1.2461
base BPB                        2.7556          2.7556
improvement (base−exp)          1.2815          1.5095

Lower fine-tuned BPB: 'mean_subword' arm.
